# Neural Models Lab: Learning, Depth, Activations, and Output Layers

Submission notebook for every task in neur_models_lab_ex.pdf.

This notebook studies XOR, connects PyTorch training to backpropagation, tests symmetry and activation behaviour, and extends the binary task to three-class classification. All experiments use CPU-only PyTorch, fixed seeds, and executable checks.

## Task 1 - Understand the problem before coding

The device has two binary sensor inputs x1 and x2. It raises a warning exactly when the sensors disagree, which is XOR.

| Input (x1, x2) | Target warning y |
|---|---:|
| (0, 0) | 0 |
| (0, 1) | 1 |
| (1, 0) | 1 |
| (1, 1) | 0 |

Input space: X = {0, 1}². Output space: Y = {0, 1}.

A coordinate sketch is shown below. The row is x2, the column is x1, and each entry is the class label:

| x2 / x1 | 0 | 1 |
|---|---:|---:|
| 0 | 0 | 1 |
| 1 | 1 | 0 |

The class-1 points are opposite corners, as are the class-0 points. No single straight
decision boundary can separate both classes, so XOR is not linearly separable.

A single affine transformation followed by sigmoid still has one linear decision boundary.
The sigmoid only converts the affine score into a probability, so an affine-only model should
not represent XOR perfectly.

Scientific claim: adding affine depth without a nonlinear hidden activation does not create
a more expressive decision boundary; the hidden nonlinearity is what enables XOR.

In [1]:
import math
import random
import numpy as np
import torch
from torch import nn

torch.set_num_threads(1)
DEVICE = torch.device("cpu")
DTYPE = torch.float32

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]], dtype=DTYPE, device=DEVICE)
y_binary = torch.tensor([[0.], [1.], [1.], [0.]], dtype=DTYPE, device=DEVICE)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
print("Shapes:", tuple(X.shape), tuple(y_binary.shape))

PyTorch: 2.14.0+cpu
Device: cpu
CUDA available: False
Shapes: (4, 2) (4, 1)


## Task 2 - Design the intelligent agent/model

Baseline architecture:

    two inputs -> Linear(2, 2) -> nonlinear activation -> Linear(2, 1) -> logit

The baseline uses sigmoid in the hidden layer, one output logit, BCEWithLogitsLoss, and
full-batch gradient-based optimisation. Sigmoid plus binary cross-entropy is appropriate
because the target is one binary event. BCEWithLogitsLoss is numerically stable.

The hidden nonlinearity is scientifically necessary because a stack of affine layers collapses
to one affine map and cannot separate XOR. The nonlinear hidden units learn intermediate
features through the chain-rule gradients supplied by backpropagation.

Evidence of learning:
1. initial loss is larger than final loss;
2. all four thresholded labels are correct;
3. the first-layer gradient is nonzero;
4. the result is reproducible with a fixed seed.

## Task 3 - LLM prompt, inspection, and implementation

Prompt used:

Generate minimal PyTorch code for the explicit XOR examples (0,0)->0, (0,1)->1,
(1,0)->1, (1,1)->0. Use exactly a 2-2-1 network with a configurable nonlinear hidden
activation, random initialisation, one output logit, BCEWithLogitsLoss, full-batch CPU
training, a fixed seed, and several thousand steps. Report initial/final loss, probabilities,
thresholded labels, and one parameter-gradient tensor after backward(). Include a zero
initialisation symmetry experiment and a three-class CrossEntropyLoss extension.

Code inspection: the forward pass is model(X); the scalar loss is made by BCEWithLogitsLoss;
loss.backward() invokes reverse-mode automatic differentiation; optimizer.step() changes
parameters. The implementation below keeps the required architecture and reports every
requested measurement.

In [2]:
def make_binary_model(hidden_activation="sigmoid"):
    activations = {"sigmoid": nn.Sigmoid(), "tanh": nn.Tanh(), "relu": nn.ReLU()}
    if hidden_activation not in activations:
        raise ValueError(f"Unknown activation: {hidden_activation}")
    return nn.Sequential(
        nn.Linear(2, 2), activations[hidden_activation], nn.Linear(2, 1)
    ).to(DEVICE)


def train_binary(hidden_activation="sigmoid", seed=17, steps=3000,
                 learning_rate=1.0, zero_initialisation=False):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    model = make_binary_model(hidden_activation)
    if zero_initialisation:
        with torch.no_grad():
            for parameter in model.parameters():
                parameter.zero_()

    loss_function = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    initial_loss = None
    early_gradient = None

    for step in range(steps):
        optimizer.zero_grad()
        loss = loss_function(model(X), y_binary)
        if step == 0:
            initial_loss = float(loss.detach())
        loss.backward()
        if step == 9:
            early_gradient = model[0].weight.grad.detach().clone()
        optimizer.step()

    with torch.no_grad():
        probabilities = torch.sigmoid(model(X)).flatten()
        predictions = (probabilities >= 0.5).to(torch.int64)
        correct = int((predictions == y_binary.flatten().to(torch.int64)).sum())

    return {
        "model": model,
        "initial_loss": initial_loss,
        "final_loss": float(loss.detach()),
        "probabilities": probabilities.detach().tolist(),
        "predictions": predictions.tolist(),
        "correct": correct,
        "early_gradient": early_gradient,
        "early_gradient_norm": float(early_gradient.norm()),
    }


baseline = train_binary()
print(f"Initial loss: {baseline['initial_loss']:.6f}")
print(f"Final loss:   {baseline['final_loss']:.6f}")
print("Probabilities:", [round(v, 6) for v in baseline["probabilities"]])
print("Predictions:  ", baseline["predictions"], "(expected [0, 1, 1, 0])")
print(f"Correct: {baseline['correct']}/4")
print(f"Early first-layer gradient norm: {baseline['early_gradient_norm']:.6f}")
print("Early first-layer gradient tensor:")
print(baseline["early_gradient"])

Initial loss: 0.693735
Final loss:   0.005549
Probabilities: [0.005339, 0.992672, 0.995143, 0.004597]
Predictions:   [0, 1, 1, 0] (expected [0, 1, 1, 0])
Correct: 4/4
Early first-layer gradient norm: 0.000294
Early first-layer gradient tensor:
tensor([[-8.6593e-05,  1.6073e-04],
        [ 1.9651e-04,  1.1989e-04]])


## Task 4A-B - Learning and backpropagation

The loss should decrease, the probabilities should be below 0.5 for (0,0) and (1,1),
above 0.5 for (0,1) and (1,0), and all four labels should be correct.

The first-layer gradient is the tensor of partial derivatives dL/dW(1). Because the loss
is a mean over four examples, it is the average of the four example-wise gradients after
the chain rule is applied. A nonzero gradient alone is not enough; decreasing loss and
correct predictions show that the signal is useful.

In [3]:
assert baseline["initial_loss"] > baseline["final_loss"]
assert baseline["predictions"] == [0, 1, 1, 0]
assert baseline["early_gradient_norm"] > 0.0
print("Basic learning checks passed.")

Basic learning checks passed.


### Task 4C - Symmetry experiment with zero initialisation

All parameters are set to zero without changing the architecture. The hidden units compute
the same values and receive the same gradients. Therefore every update preserves equality,
so the units cannot learn distinct features.

In [4]:
symmetry = train_binary(steps=10, zero_initialisation=True)
zero_weights = symmetry["model"][0].weight.detach()
print("Final hidden weight matrix:")
print(zero_weights)
print("Rows remain identical:", bool(torch.allclose(zero_weights[0], zero_weights[1])))
print("Predictions after the short symmetric run:", symmetry["predictions"])

torch.manual_seed(17)
zero_model = make_binary_model("sigmoid")
with torch.no_grad():
    for parameter in zero_model.parameters():
        parameter.zero_()
zero_loss = nn.BCEWithLogitsLoss()(zero_model(X), y_binary)
zero_loss.backward()
with torch.no_grad():
    zero_model[0].weight -= zero_model[0].weight.grad
print("Rows identical after one update:",
      bool(torch.allclose(zero_model[0].weight[0], zero_model[0].weight[1])))

Final hidden weight matrix:
tensor([[0., 0.],
        [0., 0.]])
Rows remain identical: True
Predictions after the short symmetric run: [1, 1, 1, 1]
Rows identical after one update: True


### Task 4D - Activation experiment

Only the hidden activation changes; seed, architecture, dataset, optimiser, and training
budget remain fixed. A sigmoid can have a small derivative when saturated. A ReLU has exactly
zero derivative for negative pre-activations and may become inactive. Tanh can also saturate
but is centred around zero. These results describe this four-point experiment, not a universal
ranking of activations.

In [5]:
activation_results = {}
for name in ("sigmoid", "tanh", "relu"):
    activation_results[name] = train_binary(
        hidden_activation=name, seed=17, steps=3000, learning_rate=1.0
    )

print("| Hidden activation | Final loss | 4/4 correct? | Early gradient norm |")
print("|---|---:|:---:|---:|")
for name, result in activation_results.items():
    print(f"| {name} | {result['final_loss']:.6f} | "
          f"{'Yes' if result['correct'] == 4 else 'No'} | "
          f"{result['early_gradient_norm']:.6f} |")

| Hidden activation | Final loss | 4/4 correct? | Early gradient norm |
|---|---:|:---:|---:|
| sigmoid | 0.005549 | Yes | 0.000294 |
| tanh | 0.001082 | Yes | 0.002069 |
| relu | 0.477412 | No | 0.073477 |


Interpretation: sigmoid and tanh learn all four labels in the reproducible run, while ReLU
is more sensitive to initialisation and can leave a hidden unit inactive. Gradient norm is
not a quality score by itself; final loss and predictions provide the behavioural evidence.
Inspecting pre-activations and activations distinguishes sigmoid saturation from a ReLU unit
whose pre-activation remains negative.

## Task 5 - Three-class extension

Class 0 means both sensors inactive, class 1 means disagreement, and class 2 means both
sensors active. The target indices are [0, 1, 1, 2].

The hidden representation remains two units, but the output is Linear(2, 3). Therefore the
final weight matrix has shape (3, 2), and each example has three logits. CrossEntropyLoss
takes a class index. Softmax makes nonnegative probabilities that sum to one.

For one-hot target y, the cross-entropy derivative with respect to logit z is p - y. Adding
the same constant to every logit leaves softmax unchanged, so stable implementations subtract
the maximum logit before exponentiating to prevent overflow.

In [6]:
y_multiclass = torch.tensor([0, 1, 1, 2], dtype=torch.long, device=DEVICE)


def train_multiclass(seed=17, steps=3000, learning_rate=0.05):
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(2, 2), nn.Sigmoid(), nn.Linear(2, 3)).to(DEVICE)
    loss_function = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_function(model(X), y_multiclass)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        logits = model(X)
        probabilities = torch.softmax(logits, dim=1)
        predictions = probabilities.argmax(dim=1)
    return model, loss.detach(), logits.detach(), probabilities, predictions


multiclass_model, multiclass_loss, multiclass_logits, multiclass_probabilities, multiclass_predictions = train_multiclass()
print("Output-layer weight shape:", tuple(multiclass_model[-1].weight.shape))
print("Final multiclass loss:", f"{float(multiclass_loss):.6f}")
print("Predicted classes:", multiclass_predictions.tolist(), "(expected [0, 1, 1, 2])")
print("Probabilities for all inputs:")
for row, probabilities in zip(X.tolist(), multiclass_probabilities.tolist()):
    print(tuple(int(v) for v in row), [round(v, 6) for v in probabilities])

one_vector = multiclass_probabilities[1]
print("One probability vector:", one_vector.tolist())
print("Sum:", float(one_vector.sum()))
shifted = torch.softmax(multiclass_logits[1] + 100.0, dim=0)
print("Shifted-logit probabilities:", shifted.tolist())
print("Unchanged apart from roundoff:", bool(torch.allclose(one_vector, shifted)))

assert tuple(multiclass_model[-1].weight.shape) == (3, 2)
assert multiclass_predictions.tolist() == [0, 1, 1, 2]
assert torch.isclose(one_vector.sum(), torch.tensor(1.0), atol=1e-6)
print("Three-class checks passed.")

Output-layer weight shape: (3, 2)
Final multiclass loss: 0.000132
Predicted classes: [0, 1, 1, 2] (expected [0, 1, 1, 2])
Probabilities for all inputs:
(0, 0) [0.999883, 0.000117, 0.0]
(0, 1) [6.6e-05, 0.999886, 4.8e-05]
(1, 0) [6.8e-05, 0.999885, 4.7e-05]
(1, 1) [4e-06, 0.000176, 0.99982]
One probability vector: [6.589364056708291e-05, 0.9998860359191895, 4.803355477633886e-05]
Sum: 1.0
Shifted-logit probabilities: [6.589345866814256e-05, 0.9998860359191895, 4.803355477633886e-05]
Unchanged apart from roundoff: True
Three-class checks passed.


### Think About It - Scaling to language models

Next-token prediction is still classification: one logit per class, softmax conceptually,
and cross-entropy. The p - y gradient, probability normalisation, and output/loss pairing
remain the same. Vocabulary size, embedding and hidden dimensions, sequence length,
transformer attention, memory use, batching, and computational cost change dramatically.

## Reflection questions

### 1. What did XOR demonstrate about depth and nonlinearity?

Depth by itself is not enough: a stack of affine transformations remains affine. A nonlinear
hidden activation changes the representation and allows a final linear layer to separate XOR.

### 2. What evidence showed useful backpropagation?

The first-layer gradient was nonzero, but stronger evidence was decreasing loss, correct labels,
and probabilities moving toward the correct extremes. The gradient caused useful updates.

### 3. Why did identical or zero initialisation prevent distinct features?

Identical units have identical outputs and identical chain-rule gradients. Every update
preserves equality, so the units cannot specialise.

### 4. How did activation affect the gradient?

Sigmoid and tanh can have small derivatives under saturation. ReLU has derivative zero on
negative pre-activations. The table reports an engineering observation for this seed; the
derivative mechanisms are the scientific explanation.

### 5. Why select output and loss together?

Binary events use one logit with binary cross-entropy. Mutually exclusive K-way labels use
K logits with multiclass cross-entropy. The pairing determines correct probabilities and
gradients.

### 6. What did the LLM contribute, and where was verification essential?

The LLM contributed PyTorch boilerplate, API suggestions, and diagnostic ideas. Verification
was essential for the exact architecture, labels, loss pairing, gradient measurements,
symmetry result, and final predictions.

### 7. Which tests scale and which become expensive?

Loss curves, predictions, gradient norms, activation statistics, reproducibility, and targeted
unit tests scale well. Exhaustive finite-difference gradient checks become too expensive and
should be limited to sampled parameters or small submodules.

## Responsible-use conclusion

An LLM can translate a precise design into code, but generated code is only a hypothesis.
Assertions, loss measurements, predictions, gradients, symmetry checks, and softmax checks
provide evidence that the implementation answers the scientific questions.